# LoL Match Prediction - Model Training

This notebook trains the enhanced model with new 2025-2026 data.

**Prerequisites:**
1. Upload the `tese` folder to your Google Drive
2. Run all cells in order

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Setup Project Path

**IMPORTANT:** Update the path below to match where you uploaded the project in Google Drive.

In [2]:
import os
import sys

# Update this path to your project location in Google Drive
PROJECT_PATH = '/content/drive/MyDrive/tese'

# Verify path exists
if os.path.exists(PROJECT_PATH):
    print(f"Project found at: {PROJECT_PATH}")
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, PROJECT_PATH)
    print(f"Working directory: {os.getcwd()}")
else:
    print(f"ERROR: Project not found at {PROJECT_PATH}")
    print("Please update PROJECT_PATH to match your Google Drive location")

Project found at: /content/drive/MyDrive/tese
Working directory: /content/drive/MyDrive/tese


## 3. Install Dependencies

In [3]:
!pip install -q xgboost lightgbm catboost optuna shap scikit-learn pandas numpy matplotlib seaborn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 17.6 MB/s eta 0:00:00


## 4. Verify Dataset

In [4]:
import pandas as pd

dataset_path = os.path.join(PROJECT_PATH, 'data', 'processed', 'complete_target_leagues_dataset.csv')

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded: {len(df):,} matches")
    print(f"Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"\nMatches by year:")
    print(df['year'].value_counts().sort_index())
    print(f"\nMatches by league:")
    print(df['league'].value_counts())
else:
    print(f"ERROR: Dataset not found at {dataset_path}")

Dataset loaded: 40,945 matches
Date range: 2014-01-14 17:52:02 to 2026-02-04 13:35:23

Matches by year:
year
2014    1203
2015    1342
2016    3209
2017    4307
2018    3850
2019    3594
2020    3680
2021    3954
2022    3969
2023    3941
2024    3704
2025    3684
2026     508
Name: count, dtype: int64

Matches by league:
league
LPL     13429
LCK     10058
LEC      6958
LCS      6749
WLDs     2472
MSI      1279
Name: count, dtype: int64


## 5. Check GPU Availability

In [5]:
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available - training will use CPU")
    print("Go to Runtime -> Change runtime type -> GPU to enable")

GPU available: Tesla T4
GPU memory: 15.6 GB


In [6]:
# Add scripts to Python path
import sys
import os

PROJECT_PATH = '/content/drive/MyDrive/tese'
sys.path.insert(0, PROJECT_PATH)
sys.path.insert(0, os.path.join(PROJECT_PATH, 'scripts'))

# Now import should work
from compare_results import create_baseline_from_thesis, load_baseline

# Check if baseline exists
baseline = load_baseline()
if baseline is None:
    print("Creating baseline from thesis results...")
    create_baseline_from_thesis()
else:
    print(f"Baseline exists: {baseline['description']}")
    print(f"Baseline AUC: {baseline['results']['test_auc']:.4f}")

Baseline exists: Thesis baseline (Logistic Regression, 82.97% AUC)
Baseline AUC: 0.8297


## 6. Create Baseline from Thesis Results

In [7]:
# Create baseline for comparison
from scripts.compare_results import create_baseline_from_thesis, load_baseline

# Check if baseline exists
baseline = load_baseline()
if baseline is None:
    print("Creating baseline from thesis results...")
    create_baseline_from_thesis()
else:
    print(f"Baseline exists: {baseline['description']}")
    print(f"Baseline AUC: {baseline['results']['test_auc']:.4f}")

Baseline exists: Thesis baseline (Logistic Regression, 82.97% AUC)
Baseline AUC: 0.8297


## 7. Run Training (Quick Mode for Testing)

In [8]:
# Quick test run first to verify everything works
from scripts.run_training import run_training

print("Running quick training test...")
predictor, results, comparison = run_training(
    quick_mode=True,
    use_stratified_temporal=True,
    use_enhanced_v2=True,
    use_temporal_weighting=False,
    calibrate_probs=True,
    quantify_uncertainty=False,  # Skip for quick test
    description="Quick test with 2025-2026 data"
)

Running quick training test...
LOL MATCH PREDICTION - TRAINING RUN
Timestamp: 2026-02-23T12:54:26.538704

Configuration:
  Quick mode: True
  Stratified temporal: True
  Enhanced v2 features: True
  Temporal weighting: False
  Probability calibration: True
  Uncertainty quantification: False
ULTIMATE LEAGUE OF LEGENDS MATCH PREDICTION SYSTEM
RUNNING IN QUICK MODE - Reduced training time
USING STRATIFIED TEMPORAL SPLIT - Meta-aware methodology
USING ENHANCED V2 FEATURES - Side selection, patch transition, extended interactions
⚠️ Dataset file not found at: /content/drive/MyDrive/tese/data/complete_target_leagues_dataset.csv
Looking for alternative paths...
✅ Found dataset at: /content/drive/MyDrive/tese/data/processed/complete_target_leagues_dataset.csv
Using dataset: /content/drive/MyDrive/tese/data/processed/complete_target_leagues_dataset.csv
📂 AdvancedFeatureEngineering using CLEAN dataset: /content/drive/MyDrive/tese/data/processed/complete_target_leagues_dataset.csv
✅ This dataset

PicklingError: Can't pickle <function AdvancedFeatureEngineering._analyze_pickban_strategy.<locals>.<lambda> at 0x7a41b9997f60>: it's not found as feature_engineering.advanced_feature_engineering.AdvancedFeatureEngineering._analyze_pickban_strategy.<locals>.<lambda>

## 8. Run Full Training (Takes longer)

In [ ]:
# Full training with all enhancements
from scripts.run_training import run_training

print("Running full training with all enhancements...")
print("This may take 30-60 minutes depending on hardware.")

predictor, results, comparison = run_training(
    quick_mode=False,
    use_stratified_temporal=True,
    use_enhanced_v2=True,
    use_temporal_weighting=False,
    calibrate_probs=True,
    quantify_uncertainty=True,
    description="Full training - Enhanced v2 with 2025-2026 data"
)

## 9. View Results Summary

In [ ]:
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

print(f"\nBest Model: {results['model']}")
print(f"\nKey Metrics:")
print(f"  AUC-ROC:          {results['test_auc']:.4f}")
print(f"  F1 Score:         {results['test_f1']:.4f}")
print(f"  MCC:              {results['test_mcc']:.4f}")
print(f"  Accuracy:         {results['test_accuracy']:.4f}")
print(f"  Brier Score:      {results['test_brier']:.4f}")
print(f"  ECE:              {results['test_ece']:.4f}")

print(f"\nGeneralization Gap: {results['generalization_gap']:.4f}")

# Compare to baseline
baseline = load_baseline()
if baseline:
    baseline_auc = baseline['results']['test_auc']
    improvement = results['test_auc'] - baseline_auc
    print(f"\nBaseline AUC: {baseline_auc:.4f}")
    print(f"Improvement:  {improvement:+.4f} ({improvement/baseline_auc*100:+.2f}%)")

## 10. View Training History

In [ ]:
from scripts.compare_results import list_runs

list_runs(n=5)

## 11. Save Model to Drive

In [ ]:
import joblib

# Models are already saved in the project directory
models_dir = os.path.join(PROJECT_PATH, 'models')
print(f"Models saved in: {models_dir}")

# Load metadata to show model name mapping
metadata_path = os.path.join(models_dir, 'ultimate_model_metadata.joblib')
if os.path.exists(metadata_path):
    metadata = joblib.load(metadata_path)
    saved_models = metadata.get('saved_models', {})
    print(f"\nBest model: {metadata['best_model_name']}")
    print(f"Features: {metadata['num_features']}")
    print(f"\nModel files:")
    for f in sorted(os.listdir(models_dir)):
        if f.endswith('.joblib'):
            size = os.path.getsize(os.path.join(models_dir, f)) / 1024
            model_name = saved_models.get(f, '')
            label = f"  -> {model_name}" if model_name else ""
            print(f"  {f:<40s} ({size:>8.1f} KB) {label}")
else:
    print("\nModel files:")
    for f in sorted(os.listdir(models_dir)):
        if f.endswith('.joblib'):
            size = os.path.getsize(os.path.join(models_dir, f)) / 1024
            print(f"  {f} ({size:.1f} KB)")